In [7]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# langchain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

# vectorstores
from langchain_community.vectorstores import Chroma

# utility imports
import numpy as np
from typing import List


e:\RAG LEARNING\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
# RAG Architecture Overview
print("""
RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge
""")


RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge



In [4]:
# 1. Sample data

In [19]:
#  create sample documents

sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

sample_docs


['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective f

In [20]:
# save sample docuemnt to files
import tempfile
temp_dir = tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt","w") as f:
        f.write(doc)

print(f"Sample document created in : {temp_dir}")

Sample document created in : C:\Users\hp\AppData\Local\Temp\tmpue5eygi3


In [21]:
# save sample docuemnt to files
import tempfile
temp_dir = tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(f"doc_{i}.txt","w") as f:
        f.write(doc)

print(f"Sample document created in : {temp_dir}")

Sample document created in : C:\Users\hp\AppData\Local\Temp\tmpg56bfr6v


In [8]:
# 2. Docuemtn loading

In [23]:
from langchain_community.document_loaders import DirectoryLoader,TextLoader

# Load documents fomr direcotry
loader = DirectoryLoader(
    "data",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'}

)

documents = loader.load()

print(f"Loaded {len(documents)} documents")
print(f"\n First document preview:")
print(documents[0].page_content[:200]+"...")

Loaded 3 documents

 First document preview:

    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. Ther...


In [8]:
temp_dir

'C:\\Users\\hp\\AppData\\Local\\Temp\\tmphcqy0hyr'

In [11]:
# Document splitting


In [24]:
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # Maximum size of each chunk
    chunk_overlap=50,  # Overlap between chunks to maintain context
    length_function=len,
    separators=[" "]  # Hierarchy of separators
)
chunks=text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\nChunk example:")
print(f"Content: {chunks[0].page_content[:150]}...")
print(f"Metadata: {chunks[0].metadata}")

Created 5 chunks from 3 documents

Chunk example:
Content: Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experie...
Metadata: {'source': 'data\\doc_0.txt'}


In [13]:
# Embedding Models

In [26]:
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize a simple embedding model(no api key is needed)
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)
embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 458.00it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [3]:
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [27]:
# embdeeings with open ai
sample_text = "Machine Learning is fascinating"
embeddings1 = OpenAIEmbeddings()
embeddings1

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x000001DAD9D54B90>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000001DADBFAC8F0>, model='text-embedding-ada-002', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [17]:
# vector = embeddings1.embed_query(sample_text)
# vector

In [28]:
vector = embeddings.embed_query(sample_text)
vector

[-0.04763006046414375,
 -0.09523380547761917,
 0.08622033894062042,
 0.008249763399362564,
 0.012193558737635612,
 -0.05049518495798111,
 -0.007320750504732132,
 -0.05885455012321472,
 -0.03901909664273262,
 -0.028660926967859268,
 -0.08570095151662827,
 0.07921122759580612,
 0.00468567106872797,
 -0.02150489017367363,
 -0.07088710367679596,
 0.014649437740445137,
 -0.013478824868798256,
 -0.015381768345832825,
 -0.0821557492017746,
 -0.12226202338933945,
 -0.0034172162413597107,
 0.013044723309576511,
 -0.008553974330425262,
 0.007963730953633785,
 0.024985099211335182,
 0.018824268132448196,
 0.06268894672393799,
 0.019554127007722855,
 0.04684524983167648,
 -0.06655006855726242,
 0.0034160269424319267,
 0.06802240759134293,
 -0.011118331924080849,
 0.0300443135201931,
 -0.09648734331130981,
 0.027357498183846474,
 0.009005174972116947,
 0.01710325852036476,
 0.045540425926446915,
 0.006371479015797377,
 -0.005477488040924072,
 -0.043239910155534744,
 0.03637201339006424,
 -0.0049531

In [19]:
# Initilalized the chromadb vecotr store and store the chunks in vecotr represenatation

In [29]:
chunks

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of in

In [30]:
# Correct spelling
persist_directory = "./chroma_db"

# Initialize Chroma with HuggingFace embeddings
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"),
    persist_directory=persist_directory,
    collection_name="rag_collections"
)

print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to : {persist_directory}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 342.07it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector store created with 17 vectors
Persisted to : ./chroma_db


In [ ]:
# test similarity Search

In [31]:
query="What are the types of machine learning?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, w

In [27]:
query="what is NLP?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervi

In [32]:
query="what is Deep Learning?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neura

In [29]:
print(f"Query: {query}")
print(f"\nTop {len(similar_docs)} similar chunks:")
for i, doc in enumerate(similar_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")

Query: what is Deep Learning?

Top 3 similar chunks:

--- Chunk 1 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of i...
Source: data\doc_1.txt

--- Chunk 2 ---
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are...
Source: data\doc_0.txt

--- Chunk 3 ---
Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recogn...
Source: data\doc_2.txt


In [30]:
# Advance similarity search with scores

In [33]:
result_scores = vectorstore.similarity_search_with_score(query,k=3)
result_scores

[(Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
  0.5801880359649658),
 (Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recogni

In [18]:
""" 
#### Understanding Similarity Scores
The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)


Cosine similarity (if configured):

- Higher scores = MORE similar
- Range: -1 to 1 (1 being identical)

"""

' \n#### Understanding Similarity Scores\nThe similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:\n\nChromaDB default: Uses L2 distance (Euclidean distance)\n\n- Lower scores = MORE similar (closer in vector space)\n- Score of 0 = identical vectors\n- Typical range: 0 to 2 (but can be higher)\n\n\nCosine similarity (if configured):\n\n- Higher scores = MORE similar\n- Range: -1 to 1 (1 being identical)\n\n'

In [33]:
# Initialize LLM , RAG Chain , Prompt Template,Query the RAG System

In [34]:
from langchain_community.llms import Ollama
# 1. Load DeepSeek-R1 via Ollama
llm = Ollama(model="deepseek-r1:1.5b")



In [22]:
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model= "gpt-3.5-turbo")

In [35]:
test_response=llm.invoke("What is Large Language Models")
test_response

'Large Language Models (LLMs) are advanced artificial intelligence technologies designed to generate human-like text. They can read texts, understand the context in which words appear, and produce coherent or creative responses.\n\n### Key Features of LLMs:\n1. **Context-Dependent Understanding**: LLMs learn from large amounts of text data, allowing them to understand relationships between different concepts.\n2. **Generate Text**: Once trained, these models can generate new text that mimics the style and meaning of their training data.\n3. ** creativity**: They are capable of producing original or creative responses even when given specific prompts.\n\n### Applications:\n- **Content Generation**: LLMs are used to create news articles, blog posts, and other written content on a wide range of topics.\n- **Translation and Summarization**: They can translate sentences between languages and summarize large texts into shorter versions.\n- **Documentation**: LLMs assist in writing technical 

In [25]:

# llm = ChatOpenAI(model="gpt-3.5-turbo")

# response = llm.invoke("what is llm")
# print(response.content)


In [9]:
from langchain.chat_models.base import init_chat_model

# Initialize DeepSeek-R1:1.5B via Ollama
llm = init_chat_model("ollama:deepseek-r1:1.5b")

# Test query
response = llm.invoke("What is AI?")
print(response)


content='Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that gain intelligence through learning, problem-solving, reasoning, and language understanding. The goal of AI is to create systems that can perform tasks that would require human intelligence, such as recognizing patterns, decision-making, and strategic actions.' additional_kwargs={} response_metadata={'model': 'deepseek-r1:1.5b', 'created_at': '2026-03-17T12:52:36.3967485Z', 'message': {'role': 'assistant', 'content': ''}, 'done': True, 'done_reason': 'stop', 'total_duration': 2945959400, 'load_duration': 230308200, 'prompt_eval_count': 7, 'prompt_eval_duration': 78860400, 'eval_count': 65, 'eval_duration': 2477917300} id='lc_run--019cfbda-e00b-71f0-9873-5d922827141f-0' tool_calls=[] invalid_tool_calls=[]


In [10]:
llm

ChatOllama(model='deepseek-r1:1.5b')

In [13]:
# Modern Rag chain

In [36]:
## Convert vector store to retriever
retriever=vectorstore.as_retriever(
    search_kwarg={"k":3} ## Retrieve top 3 relevant chunks
)
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001DADC2124B0>, search_kwargs={})

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain

In [13]:
## Create a prompt template
from langchain_core.prompts import ChatPromptTemplate
system_prompt="""You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [14]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [37]:
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

llm = Ollama(model="deepseek-r1:1.5b")

prompt = PromptTemplate.from_template(
"""Context:
{context}

Question:
{input}
"""
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    RunnableParallel({
        "context": retriever | format_docs,
        "input": RunnablePassthrough()
    })
    | prompt
    | llm
)

response = rag_chain.invoke("What is Deep Learning?")
print(response)

Deep Learning is a subset of machine learning that utilizes artificial neural networks composed of multiple layers, enabling hierarchical processing of data to capture complex patterns. It is particularly effective in fields such as computer vision, natural language processing (NLP), and speech recognition. Deep learning employs supervised learning, where models are trained on labeled datasets, allowing them to learn from examples to make accurate predictions. This structured approach with multiple layers processes information sequentially, enhancing the model's ability to recognize and extract features at various levels, making it transformative across diverse applications.


In [56]:
# def query_rag_modern(question, show_sources=True):
#     print(f"\n🔎 Question: {question}")
#     print("-" * 60)

#     try:
#         result = rag_chain.invoke({"input": question})

#         answer = result.get("answer", "No answer generated.")
#         context_docs = result.get("context", [])

#         print(f"\n💡 Answer:\n{answer}")

#         if show_sources and context_docs:
#             print("\n📚 Retrieved Sources:")
#             for i, doc in enumerate(context_docs, start=1):
#                 preview = doc.page_content[:200].replace("\n", " ")
#                 print(f"\n--- Source {i} ---")
#                 print(preview + "...")

#         return {
#             "question": question,
#             "answer": answer,
#             "sources": context_docs
#         }

#     except Exception as e:
#         print(f"\n❌ Error during RAG query: {str(e)}")
#         return None

In [57]:
# test_questions = [
#     "What are the three types of machine learning?",
#     "What is deep learning and how does it relate to neural networks?",
#     "What are CNNs best used for?"
# ]

# for q in test_questions:
#     query_rag_modern(q)
#     print("\n" + "="*80)

In [38]:
# Function to query the modern RAG system
def query_rag_modern(question):
    print(f"Question: {question}")
    print("-" * 50)
    
    # Using create_retrieval_chain approach
    result = rag_chain.invoke(question)
    
    print(f"Answer: {result}")
    
    return result

# Test queries
test_questions = [
    "What are the three types of machine learning?",
    "What is deep learning and how does it relate to neural networks?",
    "What are CNNs best used for?"
]

for question in test_questions:
    result = query_rag_modern(question)
    print("\n" + "="*80 + "\n")

Question: What are the three types of machine learning?
--------------------------------------------------
Answer: The three primary types of machine learning are:

1. **Supervised Learning**: Uses labeled data to train models by comparing features with target labels.

2. **Unsupervised Learning**: Finds patterns in unlabeled data without prior knowledge of outcomes.

3. **Reinforcement Learning**: Teaches systems through interaction, using rewards for success and penalties for failure.

While Deep Learning is a subset of machine learning focused on neural networks, it's not one of the main categories listed. Instead, it enhances specific areas like computer vision with techniques such as CNNs, RNNs, and Transformers.


Question: What is deep learning and how does it relate to neural networks?
--------------------------------------------------
Answer: Deep learning is a subset of machine learning that uses artificial neural networks (ANNs) inspired by the structure and function of the 

In [33]:
### Create RAG Chain Alternative - Using LCEL (LangChain Expression Language)

In [39]:
# Even more flexible approach using LCEL
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

In [40]:
# Create a custom prompt
custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question. 
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support your answer.

Context:
{context}

Question: {question}

Answer:""")
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])

In [61]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001AC90FDF560>, search_kwargs={})

In [41]:
## Format the output documents for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [42]:
## Build the chain ussing LCEL

rag_chain_lcel=(
    { 
        "context":retriever | format_docs,
        "question": RunnablePassthrough()
     }
    | custom_prompt
    | llm
    | StrOutputParser()
)

rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001DADC2124B0>, search_kwargs={})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])
| Ollama(model='deepseek-r1:1.5b')
| StrOutputParser()

In [43]:
response=rag_chain_lcel.invoke("What is Deep Learning")
response

'Deep Learning is a subset of machine learning characterized by artificial neural networks with multiple interconnected layers, which are inspired by biological neural networks. These systems are particularly effective for tasks requiring complex pattern recognition, such as computer vision and natural language processing. Specific applications include image processing (using CNNs) and facial recognition.'

In [44]:
# retriever.get_relevant_documents("What is Deep Learning")

In [70]:
# results = vectorstore.similarity_search_with_score("What is Deep Learning?", k=5)
# for doc, score in results:
#     print(score, doc.page_content[:200])


In [45]:
# Query using the LCEL approach - Fixed version
def query_rag_lcel(question):
    print(f"Question: {question}")
    print("-" * 50)
    
    # Method 1: Pass string directly (when using RunnablePassthrough)
    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")
    
    # Get source documents separately if needed
    docs = retriever.invoke(question)

    print("\nSource Documents:")
    for i, doc in enumerate(docs):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")

In [46]:
# # Test LCEL chain
print("Testing LCEL Chain:")
query_rag_lcel("What are the key concepts in reinforcement learning?")

Testing LCEL Chain:
Question: What are the key concepts in reinforcement learning?
--------------------------------------------------
Answer: The key concepts in reinforcement learning are:

- **States**: The current situation or condition of the environment.
- **Actions**: The choices an agent can make to influence the environment.
- **Rewards/Penalties**: Feedback received for each action, which guides the agent's decision-making process.
- ** Policies**: Strategies that determine the best actions to take in various states.
- **Value Functions**: Functions that estimate the cumulative reward or utility of being in a particular state or following a certain policy.

These concepts form the foundation of reinforcement learning, guiding how agents learn to make optimal decisions through interaction with their environment.

Source Documents:

--- Source 1 ---
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decision

In [73]:
query_rag_lcel("What is machine learning?")

Question: What is machine learning?
--------------------------------------------------
Answer: **Answer:**

Machine Learning is a subset of artificial intelligence that enables systems to learn and improve through experience without being explicitly programmed. It involves algorithms and models that can adapt and perform tasks based on data. The key aspects include supervised learning (using labeled data) and unsupervised learning (without labeled data), as well as reinforcement learning, which learns through interaction with an environment. Deep Learning, a subset of machine learning, utilizes neural networks inspired by the human brain's structure, enabling applications like computer vision and natural language processing.

**Answer:** Machine learning is a subset of artificial intelligence that enables systems to learn from data without explicit programming, utilizing supervised and unsupervised learning types through algorithms that adapt based on data.

Source Documents:

--- Sour

In [47]:
query_rag_lcel("What is depe learning?")

Question: What is depe learning?
--------------------------------------------------
Answer: The term "depe learning" appears to be a typo. Based on the context provided, it likely refers to reinforcement learning, which uses rewards and penalties for learning decisions through interaction with an environment.

**Answer:** Depe learning refers to reinforcement learning that uses rewards and penalties for decision-making processes.

Source Documents:

--- Source 1 ---
data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties....

--- Source 2 ---
data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties....

--- Source 3 ---
data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties....

--- Source 4 ---
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by 

In [48]:
vectorstore

In [49]:
# Add new documents to the existing vector store
new_document = """
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or penalties 
based on its actions and learns to maximize cumulative reward over time. Key concepts 
in RL include: states, actions, rewards, policies, and value functions. Popular RL 
algorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and 
Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), 
robotics, and autonomous systems.
"""

In [50]:
new_document

'\nReinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.\n'

In [51]:
chunks

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of in

In [52]:
new_doc=Document(
    page_content=new_document,
    metadata={"source": "manual_addition", "topic": "reinforcement_learning"}
)

In [53]:
new_doc

Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='\nReinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.\n')

In [63]:
## split the documents
new_chunks=text_splitter.split_documents([new_doc])
new_chunks

[Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='Reinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been'),
 Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.')]

In [64]:
### Add new documents to vectorstore
vectorstore.add_documents(new_chunks)

['075f8e54-cf88-4742-9d95-8844cbe12ff3',
 '7179510b-fc3a-4dd0-abe2-f38bb7bee75c']

In [ ]:
print(f"Added {len(new_chunks)} new chunks to the vector store")
print(f"Total vectors now: {vectorstore._collection.count()}")

Added 2 new chunks to the vector store
Total vectors now: 12


In [85]:
## query with the updated vector
new_question="What are the keys concepts in reinforcement learning"
result=query_rag_lcel(new_question)
result

Question: What are the keys concepts in reinforcement learning
--------------------------------------------------
Answer: The key concepts in reinforcement learning (RL) are:

1. **States**: These represent the current situation or condition of the environment, which an agent interacts with.

2. **Actions**: The possible choices or moves that the agent can take in each state.

3. **Rewards/Penalties**: Feedback from the environment that results from taking actions, either positive (reinforcement) or negative (punishment).

4. **Policies**: A strategy or rule that determines the sequence of actions an agent will take over time.

5. **Value Functions**: Functions used to evaluate the desirability of being in a particular state or making a specific action, helping estimate long-term benefits.

These concepts form the foundation of RL, guiding the learning process through interactions with an environment and optimizing outcomes based on rewards.

Source Documents:

--- Source 1 ---
Reinfor

In [ ]:
query_rag_lcel("what is ml")

### Advanced Rag Techniques- Conversational Memory
Understanding Conversational Memory in RAG
Conversational memory enables RAG systems to maintain context across multiple interactions. This is crucial for:

Follow-up questions that reference previous answers
Pronoun resolution (e.g., "it", "they", "that")
Context-dependent queries that build on prior discussion
Natural dialogue flow where users don't repeat context

Key Challenge:
Traditional RAG retrieves documents based only on the current query, missing important context from the conversation. For example:

User: "Tell me about Python"
Bot: explains Python programming language
User: "What are its main libraries?" ← "its" refers to Python, but retriever doesn't know this

Solution:
The modern approach uses a two-step process:

Query Reformulation: Transform context-dependent questions into standalone queries
Context-Aware Retrieval: Use the reformulated query to fetch relevant documents

In [78]:
# from langchain_community.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

In [79]:
## create a prompt that includes the chat history
contextualize_q_system_prompt = """Given a chat history and the latest user question 
which might reference context in the chat history, formulate a standalone question 
which can be understood without the chat history. Do NOT answer the question, 
just reformulate it if needed and otherwise return it as is."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

In [81]:
## create history aware retriever
# history_aware_retriever = create_history_aware_retriever(
#     llm, retriever, contextualize_q_prompt
# )
# history_aware_retriever

In [82]:
# # Create a new document chain with history
# qa_system_prompt = """You are an assistant for question-answering tasks. 
# Use the following pieces of retrieved context to answer the question. 
# If you don't know the answer, just say that you don't know. 
# Use three sentences maximum and keep the answer concise.

# Context: {context}"""

# qa_prompt = ChatPromptTemplate.from_messages([
#     ("system", qa_system_prompt),
#     MessagesPlaceholder("chat_history"),
#     ("human", "{input}"),
# ])

# question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# # Create conversational RAG chain
# conversational_rag_chain = create_retrieval_chain(
#     history_aware_retriever, 
#     question_answer_chain
# )
# print("Conversational RAG chain created!")

In [83]:
# chat_history=[]
# # First question
# result1 = conversational_rag_chain.invoke({
#     "chat_history": chat_history,
#     "input": "What is machine learning?"
# })
# print(f"Q: What is machine learning?")
# print(f"A: {result1['answer']}")

In [84]:
# chat_history.extend([
#     HumanMessage(content="What is machine learning"),
#     AIMessage(content=result1['answer'])
# ])

In [85]:
# ## Follow up question
# # Follow-up question
# result2 = conversational_rag_chain.invoke({
#     "chat_history": chat_history,
#     "input": "What are its main types?"  # Refers to ML from previous question
# })
# result2

In [1]:
# Using Grq llm

In [2]:
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

e:\RAG LEARNING\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [10]:
llm=ChatGroq(model="gemma2-9b-it",api_key=os.getenv("GROQ_API_KEY"))
llm


ChatGroq(profile={'max_input_tokens': 8192, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001E268D663F0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E268DDC080>, model_name='gemma2-9b-it', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [11]:
llm=init_chat_model(model="groq:gemma2-9b-it")
llm


ChatGroq(profile={'max_input_tokens': 8192, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001E268DDF020>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E268DDF800>, model_name='gemma2-9b-it', model_kwargs={}, groq_api_key=SecretStr('**********'))